# Notebook 04 — Win Type Classification (2025 BDB)

**Inputs:** `outputs/blocker_metrics.csv`  
**Output:** `outputs/win_types.csv`  
**Goal:** Label each of the 23,345 rush attempts with a win type (speed / power / counter / loss) using a priority cascade, then compute a context-adjusted pressure rate per player that accounts for how hard it was to rush (single-blocked vs. double-teamed).

## Imports & Paths

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
OUT_DIR = PROJECT_ROOT / "outputs"
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok = True)

## Load Inputs

In [11]:
blocker_metrics = pd.read_csv(OUT_DIR / "blocker_metrics.csv")

print(blocker_metrics.shape)
print(blocker_metrics.columns.tolist())

blocker_metrics = blocker_metrics[blocker_metrics['blocking_context'] != 'unblocked']
print(blocker_metrics.shape)

(23345, 14)
['gameId', 'playId', 'nflId', 'displayName', 'position', 'teamAbbr', 'blocking_context', 'primary_blocker_nflId', 'causedPressure', 'separation_at_15', 'separation_at_25', 'blocker_displacement', 'rusher_dir_change', 'hip_turn_delta']
(21324, 14)


## Win Type Cascade

Priority order — each play gets exactly one label:

| Priority | Label | Condition |
|---|---|---|
| 1 | `counter` | `rusher_dir_change > 45°` **and** `separation_at_25 > 0` |
| 2 | `power` | `blocker_displacement >= 6.5` |
| 3 | `speed` | `separation_at_15 > -2.0` **and** `blocker_displacement < 6.5` **and** `rusher_dir_change <= 45°` |
| 4 | `loss` | none of the above |

In [13]:
speed = (blocker_metrics['separation_at_15'] > -2.0) & (blocker_metrics['blocker_displacement'] < 6.5) & (blocker_metrics['rusher_dir_change'] <= 45)
power = (blocker_metrics['blocker_displacement'] >= 6.5)
counter = (blocker_metrics['rusher_dir_change'] > 45) & (blocker_metrics['separation_at_25'] > 0)

blocker_metrics['win_type'] = np.select(
    [counter, power, speed],
    ['Counter', 'Power', 'Speed'],
    default='Loss'
)

print(blocker_metrics['win_type'].value_counts())

Loss       9452
Speed      8356
Power      2822
Counter     694
Name: win_type, dtype: int64


## Context-Adjusted Pressure Rate

Not all rush attempts are equal — it is harder to generate pressure when double-teamed than when single-blocked. This metric gives each play a score based on how difficult the blocking situation was.

- A pressure play against a double team scores **higher** than a pressure play against one blocker
- A play with no pressure scores **0** regardless of context
- Averaged across all of a player's rushes: **1.0 = league average**, **1.5 = 50% better than average** for the blocking they faced

In [15]:
context_baseline = blocker_metrics.groupby('blocking_context')['causedPressure'].mean()
context_baseline = context_baseline.replace(0, float('nan'))

blocker_metrics['context_contribution'] = blocker_metrics['causedPressure'] / blocker_metrics['blocking_context'].map(context_baseline)

print(context_baseline)
print(blocker_metrics['context_contribution'].describe())

blocking_context
double_teamed     0.068775
single_blocked    0.116431
Name: causedPressure, dtype: float64
count    21324.000000
mean         1.000000
std          3.025157
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         14.540260
Name: context_contribution, dtype: float64


## Player Summary

Aggregate to one row per player. Include: total rushes, win type counts, adjusted pressure rate, double-team rate.  
Minimum sample filter: **≥ 30 rush attempts** for overall rankings.

In [17]:
blocker_metrics['is_speed'] = (blocker_metrics['win_type'] == 'Speed')
blocker_metrics['is_power'] = (blocker_metrics['win_type'] == 'Power')
blocker_metrics['is_counter'] = (blocker_metrics['win_type'] == 'Counter')
blocker_metrics['is_loss'] = (blocker_metrics['win_type'] == 'Loss')
blocker_metrics['is_double_teamed'] = (blocker_metrics['blocking_context'] == 'double_teamed')

player_summary = blocker_metrics.groupby(['nflId', 'displayName', 'position', 'teamAbbr']).agg(
    total_rushes = ('win_type', 'count'),
    speed_wins = ('is_speed', 'sum'),
    power_wins = ('is_power', 'sum'),
    counter_wins = ('is_counter', 'sum'),
    losses = ('is_loss', 'sum'),
    adjusted_pressure_rate = ('context_contribution', 'mean'),
    double_team_rate = ('is_double_teamed', 'mean')
).reset_index()

player_summary = player_summary[player_summary['total_rushes'] >= 30]

print(player_summary.shape)
print(player_summary.sort_values('adjusted_pressure_rate', ascending=False).head(10))

(167, 11)
     nflId       displayName position teamAbbr  total_rushes  speed_wins  \
63   44813     Myles Garrett       DE      CLE           190          40   
167  52585     Michael Danna       DE       KC            71          53   
46   42465   Za'Darius Smith       DE      MIN           196          60   
10   37145    Justin Houston      OLB      BAL           118          25   
76   44915  Trey Hendrickson       DE      CIN           192          48   
115  47785         Nick Bosa       DE       SF           141          28   
232  54552    Cameron Thomas      OLB      ARI            55          19   
184  53447   Jaelan Phillips      OLB      MIA           195          66   
182  53441     Micah Parsons      OLB      DAL           171          42   
136  47944   Charles Omenihu       DE       SF           136          76   

     power_wins  counter_wins  losses  adjusted_pressure_rate  \
63           51             8      91                2.138839   
167           8        

## Validation

In [18]:
print(pd.crosstab(blocker_metrics['blocking_context'], blocker_metrics['win_type']))

print(blocker_metrics.groupby('win_type')['causedPressure'].mean().round(3))

print(blocker_metrics.groupby('blocking_context')['causedPressure'].mean().round(3))

win_type          Counter  Loss  Power  Speed
blocking_context                             
double_teamed         319  1766    415   3098
single_blocked        375  7686   2407   5258
win_type
Counter    0.045
Loss       0.112
Power      0.159
Speed      0.082
Name: causedPressure, dtype: float64
blocking_context
double_teamed     0.069
single_blocked    0.116
Name: causedPressure, dtype: float64


## Save Output

In [20]:
blocker_metrics.to_csv(OUT_DIR / "win_types.csv", index=False)
player_summary.to_csv(OUT_DIR / "player_win_type_summary.csv", index=False)

print("Shape:", blocker_metrics.shape)
print("Columns:", blocker_metrics.columns.tolist())
print("Unique rushers:", blocker_metrics['nflId'].nunique())

Shape: (21324, 21)
Columns: ['gameId', 'playId', 'nflId', 'displayName', 'position', 'teamAbbr', 'blocking_context', 'primary_blocker_nflId', 'causedPressure', 'separation_at_15', 'separation_at_25', 'blocker_displacement', 'rusher_dir_change', 'hip_turn_delta', 'win_type', 'context_contribution', 'is_speed', 'is_power', 'is_counter', 'is_loss', 'is_double_teamed']
Unique rushers: 250
